# Tutorial: End-to-End Pipeline

Complete TALON workflow from model definition through hardware simulation.

# 1. Define the Model


In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

class ConvSNN(nn.Module):
    """Conv feature extractor with spiking classifier."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 14 * 14, 128)
        self.lif1 = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(128, 10)
        self.lif2 = snn.Leaky(beta=0.9)

    def forward(self, x, mem1=None, mem2=None):
        if mem1 is None:
            mem1 = self.lif1.init_leaky()
        if mem2 is None:
            mem2 = self.lif2.init_leaky()
        x = self.pool(self.relu(self.conv1(x)))
        x = self.fc1(self.flatten(x))
        spk1, mem1 = self.lif1(x, mem1)
        x = self.fc2(spk1)
        spk2, mem2 = self.lif2(x, mem2)
        return spk2, mem1, mem2

model = ConvSNN()
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

# 2. Export to TALON IR


In [ ]:
from snntorch.export_talonir import export_to_ir
from talon import ir

graph = export_to_ir(model, torch.randn(1, 1, 28, 28))

print(f"Exported: {len(graph.nodes)} nodes, {len(graph.edges)} edges")
for name, node in graph.nodes.items():
    print(f"  {name:10s} -> {type(node).__name__}")

# Serialize


In [ ]:
import tempfile, os

tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "conv_snn.t1c")
ir.write(path, graph)
print(f"Saved: {os.path.getsize(path):,} bytes")

# 3. Graph Analysis


In [ ]:
from talon import sdk

stats = sdk.analyze_graph(graph)

print(f"Parameters:  {stats.total_params:,}")
print(f"Memory:      {stats.total_bytes:,} bytes")
print(f"FLOPs:       {stats.total_flops:,}")
print(f"Depth:       {stats.depth}")
print(f"Width:       {stats.width}")
print(f"LIF neurons: {stats.lif_count}")
print(f"Conv layers: {stats.conv_count}")
print(f"Types:       {stats.type_counts}")

# 4. Hardware Profile


In [ ]:
prof = sdk.profile_graph(graph)

print(f"Weight memory:     {prof.weight_memory:,} bytes")
print(f"Activation memory: {prof.activation_memory:,} bytes")
print(f"State memory:      {prof.state_memory:,} bytes")
print(f"Total memory:      {prof.total_memory:,} bytes")
print(f"MAC ops:           {prof.mac_ops:,}")
print(f"Spike ops:         {prof.spike_ops:,}")
print(f"Largest layer:     {prof.largest_layer}")
print(f"Recommendations:   {prof.recommendations}")

# 5. Linting & Fingerprinting


In [ ]:
lint_result = sdk.lint_graph(graph)
print(f"Lint issues: {len(lint_result.issues)}")
for issue in lint_result.issues:
    print(f"  [{issue.severity.value}] {issue.code}: {issue.message}")

fp = sdk.fingerprint_graph(graph)
print(f"\nFingerprint: {fp}")

# 6. Weight Quantization


In [ ]:
qgraph = sdk.quantize_weights(graph, bits=8, per_channel=True)

prof_q = sdk.profile_graph(qgraph)
print(f"FP32 weight memory: {prof.weight_memory:,} bytes")
print(f"INT8 weight memory: {prof_q.weight_memory:,} bytes")

# 7. Hardware Partitioning


In [ ]:
from talon.graph import partition, HardwareSpec, allocate, place

hw = HardwareSpec.zynq_us_plus()
hw.validate()

print(f"Cores:          {hw.num_cores}")
print(f"Neurons/core:   {hw.max_neurons_per_core}")
print(f"SRAM/core:      {hw.sram_bytes_per_core:,} bytes")
print(f"Feedback delay: {hw.max_feedback_delay}")

# Partition, Allocate, Place


In [ ]:
partitioned = partition(graph, hw)
pm = partitioned.partition_metadata

print(f"Algorithm:  {pm['algorithm']}")
print(f"Cores used: {pm['num_cores_used']}/{hw.num_cores}")

resources = allocate(partitioned, hw)
print(f"\nTotal weight bytes:    {resources.total_weight_bytes:,}")
print(f"Total state bytes:     {resources.total_state_bytes:,}")
print(f"Peak core utilization: {resources.peak_core_utilization:.2f}")
print(f"Fits hardware:         {resources.fits_hardware}")

placement = place(partitioned, hw)
print(f"\nMapping:          {placement.logical_to_physical}")
print(f"Total hop distance: {placement.total_hop_distance}")
print(f"Improvement:       {placement.improvement:.1f}%")

# 8. CPU Simulation


In [ ]:
from talon.backend import get_backend

cpu = get_backend("cpu")

sim = cpu.simulate(graph, n_steps=10)
print(f"Timesteps:    {sim.timesteps_run}")
print(f"Spike counts: {sim.spike_counts}")
print(f"Output valid: {sim.outputs_valid}")

# 9. Energy Profiling


In [ ]:
profile_result = cpu.profile(graph, n_steps=10, energy_preset="45nm_cmos")

print(f"Total latency:  {profile_result.total_latency_us:.1f} us")
print(f"Total energy:   {profile_result.energy_estimate_uj:.4f} uJ")
print(f"  MAC energy:   {profile_result.mac_energy_uj:.4f} uJ")
print(f"  Spike energy: {profile_result.spike_energy_uj:.4f} uJ")
print(f"  SRAM energy:  {profile_result.sram_energy_uj:.4f} uJ")
print(f"Peak memory:    {profile_result.peak_memory_bytes:,} bytes")
print(f"Neuron util:    {profile_result.neuron_utilization}")

# 10. Visualization


In [ ]:
graph_html = sdk.export_html(graph, os.path.join(tmpdir, "graph.html"))
print(f"Graph:      {os.path.getsize(graph_html):,} bytes")

part_html = sdk.visualize_partitioned(
    partitioned, os.path.join(tmpdir, "partitions.html")
)
print(f"Partitions: {os.path.getsize(part_html):,} bytes")

sched_html = sdk.visualize_execution_schedule(
    graph, os.path.join(tmpdir, "schedule.html")
)
print(f"Schedule:   {os.path.getsize(sched_html):,} bytes")

# 11. Event Encoding


In [ ]:
from talon.io.encoding import rate_encode, latency_encode, delta_encode

signal = np.random.rand(32).astype(np.float32)

spikes_rate = rate_encode(signal, n_steps=100)
print(f"Rate encoding:    {spikes_rate.shape} -> {int(spikes_rate.sum())} spikes")

spikes_lat = latency_encode(signal, n_steps=100, tau=5.0)
print(f"Latency encoding: {spikes_lat.shape} -> {int(spikes_lat.sum())} spikes")

sequence = np.random.rand(50, 32).astype(np.float32)
spikes_delta = delta_encode(sequence, threshold=0.1)
print(f"Delta encoding:   {spikes_delta.shape} -> {int(np.abs(spikes_delta).sum())} spikes")

# 12. Event I/O (HDF5)


In [ ]:
from talon.io.h5 import H5EventWriter, H5EventReader, EVENT_DTYPE

events = np.zeros(1000, dtype=EVENT_DTYPE)
events['t'] = np.sort(np.random.randint(0, 100_000, size=1000))
events['x'] = np.random.randint(0, 128, size=1000)
events['y'] = np.random.randint(0, 128, size=1000)
events['p'] = np.random.randint(0, 2, size=1000)

event_path = os.path.join(tmpdir, "events.h5")
with H5EventWriter(event_path) as w:
    w.write(events)
print(f"Wrote {len(events)} events")

reader = H5EventReader(event_path)
loaded = reader.read_all()
print(f"Read back: {len(loaded)} events")

window = reader.read_time_window(10_000, 50_000)
print(f"Time window [10k, 50k]: {len(window)} events")

# 13. Throughput Benchmark


In [ ]:
from talon.io.throughput import benchmark_throughput

result = benchmark_throughput(n_events=100_000, include_encoding=True)
print(f"Events/sec:   {result.events_per_sec:,.0f}")
print(f"RGB FPS:      {result.rgb_fps:.1f}")
print(f"Event FPS:    {result.event_fps:.1f}")
print(f"Encoding FPS: {result.encoding_fps:.1f}")

# 14. Full Pipeline (One Call)


In [ ]:
pipeline_result = sdk.run_pipeline(
    path,
    sdk.PipelineConfig(
        target="cpu",
        timesteps=10,
        quantize=False,
        partition=False,
        lint=True,
        profile=True,
    )
)

print(f"Errors:     {pipeline_result.errors}")
print(f"Lint:       {len(pipeline_result.lint_results)} issues")
if pipeline_result.analysis:
    print(f"Analysis:   {pipeline_result.analysis.total_params:,} params, depth={pipeline_result.analysis.depth}")
if pipeline_result.profile:
    print(f"Profile:    latency={pipeline_result.profile.total_latency_us:.1f}us, energy={pipeline_result.profile.energy_estimate_uj:.4f}uJ")
if pipeline_result.simulation:
    print(f"Simulation: {pipeline_result.simulation.timesteps_run} steps, spikes={pipeline_result.simulation.spike_counts}")